# CCE PoC v2: 80-task, 4-repo ablation

Scales the v1 PoC (n=20, httpx only) to **n=80 across httpx + Flask + FastAPI + Requests** (20 each, 10 easy + 10 hard per repo). The same 6-arm ablation. Goal: a tighter, less-noisy answer to whether contrastive-code-entropy features add signal beyond V6's hidden-state probe.

**Compute:** Colab Pro (A100 strongly recommended). ~60 minutes wall-clock.

**Output:** `research/paper/cce_poc_v2_results.json` plus per-repo breakdowns.

## 1. Install + clone repos

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

## 2. HuggingFace login (CodeLlama is gated)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

## 3. Mount Drive (for resumable cache)

In [ ]:
MOUNT_DRIVE = True
RESULTS_PATH  = 'research/paper/cce_poc_v2_results.json'
CACHE_PATH    = 'research/paper/cce_poc_v2_features.json'
PHASE3_CACHE  = 'research/paper/cce_poc_v2_phase3.json'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/cce_poc_v2'
    !mkdir -p {drive_dir}
    RESULTS_PATH = f'{drive_dir}/cce_poc_v2_results.json'
    CACHE_PATH   = f'{drive_dir}/cce_poc_v2_features.json'
    PHASE3_CACHE = f'{drive_dir}/cce_poc_v2_phase3.json'
    print(f'Will write to {RESULTS_PATH}')

## 4. Run the v2 PoC

The script auto-clones httpx / Flask / FastAPI / Requests under `/content/`. Phase 1 runs ~20 minutes on A100; Phase 3 with the shared cache runs ~30 minutes. Total ≈1h.

Resumable: rerun with `--skip-generation` if Phase 1 already completed.

In [ ]:
%cd /content/reposynth
!python /content/reposynth/research/paper/cce_poc_v2.py \
    --repos-dir /content \
    --out {RESULTS_PATH} \
    --features-cache {CACHE_PATH} \
    --phase3-cache {PHASE3_CACHE}

## 5. Inspect results

In [ ]:
import json
with open(RESULTS_PATH) as f:
    res = json.load(f)

print(f"n_tasks = {res['n_tasks']}, repos = {res['repos']}")
print()
print('PHASE 2: classifier discriminative power (LOO CV, n=80)')
print(f"{'arm':<20} {'#feat':>5} {'acc':>5} {'prec':>5} {'rec':>5} {'f1':>5}")
for arm, r in res['phase2_loo_ablation'].items():
    print(f"{arm:<20} {r['n_features']:>5d} {r['accuracy']:>5.3f} "
          f"{r['precision']:>5.3f} {r['recall']:>5.3f} {r['f1']:>5.3f}")

if res.get('phase3_end_to_end'):
    print()
    print('PHASE 3: end-to-end with retrieval gating')
    print(f"{'arm':<20} {'init':>5} {'final':>5} {'always':>5} {'used':>5} {'saved':>5} {'save%':>6}")
    for arm, r in res['phase3_end_to_end'].items():
        print(f"{arm:<20} {r['initial_accuracy']:>5.3f} {r['final_accuracy']:>5.3f} "
              f"{r['always_retrieve_accuracy']:>5.3f} {r['n_retrievals_used']:>5d} "
              f"{r['n_retrievals_saved']:>5d} {r['retrieval_save_rate']*100:>5.1f}%")

print()
print('PHASE 2 PER-REPO F1:')
print(f"{'arm':<20} {'httpx':>7} {'flask':>7} {'fastapi':>8} {'requests':>9}")
for arm, by_repo in res['phase2_per_repo'].items():
    parts = [f"{arm:<20}"]
    for repo in ['httpx', 'flask', 'fastapi', 'requests']:
        f1 = by_repo.get(repo, {}).get('f1', 0.0)
        parts.append(f"{f1:>7.3f}" if repo != 'fastapi' and repo != 'requests' else f"{f1:>{8 if repo=='fastapi' else 9}.3f}")
    print(' '.join(parts))

p2 = res['phase2_loo_ablation']
delta_f1 = p2['v6_plus_cce']['f1'] - p2['v6_full']['f1']
delta_acc = p2['v6_plus_cce']['accuracy'] - p2['v6_full']['accuracy']
print()
print(f'HEADLINE Δ (v6_plus_cce − v6_full):  Δacc={delta_acc:+.3f}  Δf1={delta_f1:+.3f}')
if delta_f1 > 0.05:
    print('  → CCE features carry signal at n=80. The paper has a defensible novelty story.')
elif delta_f1 > 0.01:
    print('  → CCE features add small signal at n=80. Pareto-frame the paper.')
elif delta_f1 > -0.02:
    print('  → CCE features are within noise. Pivot to saplma-vs-flare framing.')
else:
    print('  → CCE features hurt. Drop CCE; pitch around the saplma-vs-flare finding.')

saplma = p2['saplma_only']['f1']
flare  = p2['flare_only']['f1']
print()
print(f'SECONDARY  saplma_only F1={saplma:.3f}  vs  flare_only F1={flare:.3f}  Δ={flare-saplma:+.3f}')
if flare - saplma > 0.10:
    print('  → Plain-entropy beats hidden-state probes substantially. This is its own paper finding.')
elif abs(flare - saplma) <= 0.05:
    print('  → saplma and flare are comparable; standard hallucination-detection territory.')